# Assignment: Real-Time Data Analytics with Apache Spark and PostgreSQL

This notebook joins real-time streaming events from the `music-fhtw` Kafka topic with static
reference data (tracks, genres, albums, artists) from the `music_store` PostgreSQL database,
to build genre-based, gender-based and album-based "Top 20" playlists.

**Student ID used for table naming:** `ds25m046` (replace with your own).

## 1. Environment Setup

We configure paths and settings for the PostgreSQL JDBC driver and the Kafka connection,
exactly as we did in the previous sink/streaming notebooks.

In [ ]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL (music_store database)
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

# Schema in which we will create our result tables
schema_name = "streaming_data"

# Our student ID, used as a prefix for all tables we create
student_id = "ds25m046"  # Replace with your actual student ID

## 2. Configure Kafka

We subscribe to the `music-fhtw` topic, which streams the real-time song-play events
of our music streaming service.

In [ ]:
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
    "kafka.bootstrap.servers": "46.225.20.89:9092",
    "subscribe": "music-fhtw",
    "kafka.security.protocol": "PLAINTEXT",
    "startingOffsets": "latest",
}

## 3. Create the Spark Session

As before, we configure the Spark session to include both the Kafka connector
package and the PostgreSQL JDBC driver, so we can read/write from Kafka and PostgreSQL
in the same job.

In [ ]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use Kafka + the JDBC jar
spark = SparkSession.builder \
    .appName("Music Streaming Analytics") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

# Reduce the number of shuffle partitions for our small local cluster
spark.conf.set("spark.sql.shuffle.partitions", "4")

## 4. Read and Parse the Kafka Stream

We read the raw Kafka stream, cast the message value to a string, and then parse the
JSON payload using the schema of the music log events (same schema as in notebook 3).

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, FloatType

# Read the raw streaming data from Kafka
raw_df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Cast the key/value columns to strings
df_message = raw_df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

In [ ]:
# Define the schema for the JSON data coming from Kafka
json_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("auth", StringType(), True),
    StructField("page", StringType(), True),
    StructField("song", StringType(), True),
    StructField("level", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("userId", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("location", StringType(), True),
    StructField("track_id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("sessionId", IntegerType(), True),
    StructField("userAgent", StringType(), True),
    StructField("registration", LongType(), True),
    StructField("itemInSession", IntegerType(), True)
])

# Parse the JSON payload and select all fields as top-level columns
df_value = df_message.selectExpr("CAST(value AS STRING) as json_string")
parsed_df = df_value.withColumn("jsonData", from_json(col("json_string"), json_schema)).select("jsonData.*")

## 5. Load Static Reference Data from PostgreSQL

We need three pieces of static information from the `music_store` database:

1. **Genre per track** – to build the genre-based playlists (Task 1).
2. **Track / artist names** – to enrich every playlist with human-readable track and artist info.
3. **Album / artist info per track** – to build the Top 20 albums table (Task 3).

We load each of these as small lookup DataFrames via JDBC and join the stream against them.

In [ ]:
# Track -> genre lookup (used for the genre playlists in Task 1)
df_genres = spark.read.jdbc(
    url=url,
    table="""(
        SELECT tracks.id AS track_id, g.name AS genre
        FROM tracks
        JOIN public.genres g ON g.id = tracks.genre_id
    ) AS genre_data""",
    properties=postgres_options
)

df_genres.show(10)

In [ ]:
# Track -> track name + artist name lookup (used to enrich every playlist)
df_tracks = spark.read.jdbc(
    url=url,
    table="""(
        SELECT tracks.id AS track_id,
               tracks.name AS track_name,
               artists.name AS artist_name
        FROM tracks
        JOIN albums ON albums.id = tracks.album_id
        JOIN artists ON artists.id = albums.artist_id
    ) AS track_data""",
    properties=postgres_options
)

df_tracks.show(10)

In [ ]:
# Track -> album + artist lookup (used for the Top 20 albums table in Task 3)
df_albums = spark.read.jdbc(
    url=url,
    table="""(
        SELECT tracks.id AS track_id,
               albums.id AS album_id,
               albums.title AS album_name,
               artists.name AS artist_name
        FROM tracks
        JOIN albums ON albums.id = tracks.album_id
        JOIN artists ON artists.id = albums.artist_id
    ) AS album_data""",
    properties=postgres_options
)

df_albums.show(10)

## 6. Join the Streaming Data with the Static Data

We join the parsed streaming events with the `df_genres` and `df_tracks` lookup
DataFrames on `track_id`. The result, `df_enriched`, contains every streaming
event enriched with genre, track name and artist name, and is the basis for all
playlists in Tasks 1 and 2.

In [ ]:
# Join the streaming data with genre information
df_with_genre = parsed_df.join(df_genres, parsed_df.track_id == df_genres.track_id, "inner") \
                          .drop(df_genres.track_id)

# Join the result with track/artist name information
df_enriched = df_with_genre.join(df_tracks, df_with_genre.track_id == df_tracks.track_id, "inner") \
                            .drop(df_tracks.track_id)

# df_enriched now has: ..., genre, track_name, artist_name (plus all original streaming columns)

## 7. Helper Function: Writing Streaming Aggregations to PostgreSQL

We define a reusable `foreachBatch` writer that writes the current micro-batch
(in "complete" output mode this is the full up-to-date Top-20 table) to a given
PostgreSQL table, overwriting it each time so the table always reflects the
current Top 20.

In [ ]:
def write_top20_to_postgres(table_name):
    """Returns a foreachBatch function that overwrites `table_name`
    with the (already limited to 20 rows) contents of the batch."""
    def inner(df, epoch_id):
        df.write \
            .format("jdbc") \
            .options(**postgres_options) \
            .option("dbtable", table_name) \
            .mode("overwrite") \
            .save()
    return inner

## Task 1: Genre-Based Playlists (Pop, Jazz, Rock)

For each of the three genres, we:

1. Filter the enriched stream to that genre.
2. Aggregate by `track_id`, `track_name`, `artist_name` and count plays.
3. Order by play count (descending) and keep the Top 20.
4. Write the result to `streaming_data.<student_id>_playlist_<genre>` using `foreachBatch`,
   overwriting the table on every micro-batch so it always reflects the current Top 20.

In [ ]:
from pyspark.sql.functions import desc

def build_top20_playlist_query(genre_name, table_suffix):
    """Build and start a streaming query that maintains the Top 20 most played
    tracks for a given genre in a PostgreSQL table."""

    # Filter the enriched stream to the requested genre
    df_genre = df_enriched.filter(df_enriched.genre == genre_name)

    # Count plays per track
    playlist_df = (df_genre
                    .groupBy("track_id", "track_name", "artist_name")
                    .count()
                    .withColumnRenamed("count", "play_count")
                    .orderBy(desc("play_count"))
                    .limit(20))

    table_name = f"{schema_name}.{student_id}_playlist_{table_suffix}"
    print(f"Starting playlist query for genre '{genre_name}' -> table {table_name}")

    query = playlist_df.writeStream \
        .outputMode("complete") \
        .foreachBatch(write_top20_to_postgres(table_name)) \
        .start()

    return query

In [ ]:
# Start the three genre-based playlist queries
query_pop = build_top20_playlist_query("Pop", "pop")
query_jazz = build_top20_playlist_query("Jazz", "jazz")
query_rock = build_top20_playlist_query("Rock", "rock")

In [ ]:
# Let the queries run for a while to accumulate streaming events
import time
time.sleep(60)

# Stop the genre playlist queries
for q in [query_pop, query_jazz, query_rock]:
    q.stop()

## Task 2: Gender-Specific Playlists

We build:

1. An overall Top 20 playlist for **male** listeners (`<student_id>_playlist_male`).
2. An overall Top 20 playlist for **female** listeners (`<student_id>_playlist_female`).
3. One additional genre+gender playlist of choice, e.g. **male Rock listeners**
   (`<student_id>_playlist_male_rock`).

All three reuse the same `build_top20_playlist_query`-style logic, but filter on
`gender` (and optionally `genre`) instead of / in addition to genre.

In [ ]:
def build_top20_gender_query(gender_value, table_suffix, genre_name=None):
    """Build and start a streaming query that maintains the Top 20 most played
    tracks for a given gender (and, optionally, a given genre) in a PostgreSQL table."""

    df_filtered = df_enriched.filter(df_enriched.gender == gender_value)

    if genre_name is not None:
        df_filtered = df_filtered.filter(df_filtered.genre == genre_name)

    playlist_df = (df_filtered
                    .groupBy("track_id", "track_name", "artist_name")
                    .count()
                    .withColumnRenamed("count", "play_count")
                    .orderBy(desc("play_count"))
                    .limit(20))

    table_name = f"{schema_name}.{student_id}_playlist_{table_suffix}"
    print(f"Starting playlist query for gender '{gender_value}'"
          + (f" / genre '{genre_name}'" if genre_name else "")
          + f" -> table {table_name}")

    query = playlist_df.writeStream \
        .outputMode("complete") \
        .foreachBatch(write_top20_to_postgres(table_name)) \
        .start()

    return query

In [ ]:
# Overall Top 20 for male listeners ("M") and female listeners ("F")
query_male = build_top20_gender_query("M", "male")
query_female = build_top20_gender_query("F", "female")

# Bonus: Top 20 Rock tracks among male listeners
query_male_rock = build_top20_gender_query("M", "male_rock", genre_name="Rock")

In [ ]:
# Let the queries run for a while to accumulate streaming events
import time
time.sleep(60)

# Stop the gender-based playlist queries
for q in [query_male, query_female, query_male_rock]:
    q.stop()

## Task 3: Top 20 Albums

We aggregate the enriched stream by `album_id`, counting how many tracks were
played from each album, then take the Top 20. The result is written to
`streaming_data.<student_id>_top_20_albums` with columns
`album_id`, `album_name`, `artist_name`.

In [ ]:
# Join the streaming data with album/artist information
df_with_album = parsed_df.join(df_albums, parsed_df.track_id == df_albums.track_id, "inner") \
                          .drop(df_albums.track_id)

# Count plays per album, keeping album name + artist name
album_playcount_df = (df_with_album
                       .groupBy("album_id", "album_name", "artist_name")
                       .count()
                       .withColumnRenamed("count", "play_count")
                       .orderBy(desc("play_count"))
                       .limit(20)
                       .select("album_id", "album_name", "artist_name", "play_count"))

table_name_albums = f"{schema_name}.{student_id}_top_20_albums"
print(f"Starting Top 20 albums query -> table {table_name_albums}")

query_albums = album_playcount_df.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_top20_to_postgres(table_name_albums)) \
    .start()

In [ ]:
# Let the query run for a while to accumulate streaming events
import time
time.sleep(60)

query_albums.stop()

## 8. Cleanup

Always stop all active streaming queries and the Spark session at the end of the
notebook, to avoid leaving background jobs running that could flood the database.

In [ ]:
# Stop any remaining active streams
for s in spark.streams.active:
    s.stop()

In [ ]:
spark.stop()